# Fraud Detection Analysis

This project simulates fraud detection in transactional data.

## Contents:
- Data generation
- Exploratory analysis
- Feature engineering
- Modeling
- Business evaluation

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

In [ ]:
# 1. PARÁMETROS
# -----------------------------
N_USERS = 1000
N_TRANSACTIONS = 50000
FRAUD_RATE = 0.02


In [ ]:
# 2. GENERAR USUARIOS
# -----------------------------
users = pd.DataFrame({
    "user_id": np.arange(1, N_USERS + 1),
    "pais": np.random.choice(["AR", "BR", "MX"], N_USERS, p=[0.6, 0.2, 0.2]),
    "edad_cuenta_dias": np.random.randint(10, 1000, N_USERS),
    "promedio_monto": np.random.uniform(1000, 20000, N_USERS),
    "frecuencia_compra": np.random.uniform(0.5, 3, N_USERS)
})

In [ ]:
# 3. GENERAR TRANSACCIONES
# -----------------------------
transactions = pd.DataFrame({
    "transaction_id": np.arange(1, N_TRANSACTIONS + 1),
    "user_id": np.random.choice(users["user_id"], N_TRANSACTIONS),
    "timestamp": pd.to_datetime("2025-01-01") + pd.to_timedelta(
        np.random.randint(0, 60*60*24*30, N_TRANSACTIONS), unit="s"
    ),
    "device": np.random.choice(["mobile", "desktop"], N_TRANSACTIONS, p=[0.7, 0.3])
})

In [ ]:
# Merge con info de usuarios
transactions = transactions.merge(users, on="user_id", how="left")

# Generar monto basado en el promedio del usuario
transactions["monto"] = np.random.normal(
    transactions["promedio_monto"],
    transactions["promedio_monto"] * 0.3
)

transactions["monto"] = transactions["monto"].clip(100, None)

print(transactions.head())
print(transactions.columns)

In [ ]:
# País igual al del usuario (normal)
transactions["pais_tx"] = transactions["pais"]

In [ ]:
# Inicialmente todo no fraude
transactions["es_fraude"] = 0

In [ ]:
# 4. INYECTAR FRAUDE
# -----------------------------
n_fraud = int(N_TRANSACTIONS * FRAUD_RATE)
fraud_idx = np.random.choice(transactions.index, n_fraud, replace=False)

In [ ]:
# Caso 1: monto muy alto
transactions.loc[fraud_idx[:int(n_fraud*0.4)], "monto"] *= 5

In [ ]:
# Caso 2: cambio de país
transactions.loc[fraud_idx[int(n_fraud*0.4):int(n_fraud*0.7)], "pais_tx"] = np.random.choice(["US", "CN", "IN"])

In [ ]:
# Caso 3: cuenta nueva + monto alto
new_account_idx = fraud_idx[int(n_fraud*0.7):]
transactions.loc[new_account_idx, "edad_cuenta_dias"] = np.random.randint(1, 5, len(new_account_idx))
transactions.loc[new_account_idx, "monto"] *= 4

In [ ]:
# Marcar fraude
transactions.loc[fraud_idx, "es_fraude"] = 1

In [ ]:
transactions.groupby("es_fraude")["monto"].mean()

In [ ]:
transactions.groupby("es_fraude")["edad_cuenta_dias"].mean()

In [ ]:
transactions["pais_tx"].value_counts()

In [ ]:
# 1. KPIs DE FRAUDE

# Tasa de fraude total
fraud_rate = transactions["es_fraude"].mean()

# Promedio de monto en fraude vs no fraude
avg_monto_fraud = transactions[transactions["es_fraude"] == 1]["monto"].mean()
avg_monto_legit = transactions[transactions["es_fraude"] == 0]["monto"].mean()

# Pérdida total estimada por fraude
fraud_loss = transactions.loc[transactions["es_fraude"] == 1, "monto"].sum()

print("Fraud rate:", fraud_rate)
print("Avg monto (fraude):", avg_monto_fraud)
print("Avg monto (no fraude):", avg_monto_legit)
print("Pérdida total por fraude:", fraud_loss)

In [ ]:
# 2. ANÁLISIS DE PATRONES

# Tasa de fraude por país de la transacción
fraud_by_country = transactions.groupby("pais_tx")["es_fraude"].mean()

# Tasa de fraude por tipo de dispositivo
fraud_by_device = transactions.groupby("device")["es_fraude"].mean()

# Crear variable de monto alto
transactions["high_monto"] = transactions["monto"] > 10000

# Tasa de fraude según monto alto
fraud_by_monto = transactions.groupby("high_monto")["es_fraude"].mean()

print("\nFraude por país:\n", fraud_by_country)
print("\nFraude por dispositivo:\n", fraud_by_device)
print("\nFraude por monto alto:\n", fraud_by_monto)

In [ ]:
print(transactions.columns)

In [ ]:
# 3. FEATURE ENGINEERING (SEÑALES DE RIESGO)

# Ratio entre monto de la transacción y el promedio del usuario
transactions["ratio_monto"] = transactions["monto"] / transactions["promedio_monto"]

# Flag de cuenta nueva (menos de 7 días)
transactions["cuenta_nueva"] = transactions["edad_cuenta_dias"] < 7

# Inconsistencia de país (usuario vs transacción)
transactions["pais_diff"] = transactions["pais"] != transactions["pais_tx"]

In [ ]:
# 4. MODELO SIMPLE (REGRESIÓN LOGÍSTICA)
# =====================================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Selección de features
features = ["ratio_monto", "cuenta_nueva", "pais_diff"]

X = transactions[features]
y = transactions["es_fraude"]

In [ ]:
# División en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entrenamiento del modelo
model = LogisticRegression()
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Evaluación
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred))


In [ ]:
# 5. SCORE Y DECISIÓN DE NEGOCIO
# =====================================================

# Probabilidad de fraude (score)
transactions["score"] = model.predict_proba(X)[:, 1]

# Definir umbral de decisión (ajustable)
threshold = 0.7

# Flag de transacciones sospechosas
transactions["flag"] = transactions["score"] > threshold


In [ ]:
#Evaluación de impacto

# Fraudes detectados correctamente
fraud_detected = transactions[
    (transactions["es_fraude"] == 1) & (transactions["flag"] == True)
].shape[0]

# Falsos positivos (usuarios legítimos bloqueados)
false_positives = transactions[
    (transactions["es_fraude"] == 0) & (transactions["flag"] == True)
].shape[0]

print("\nFraudes detectados:", fraud_detected)
print("Falsos positivos:", false_positives)